In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import (
    col,
    explode,
    to_timestamp,
    from_unixtime,
    to_date,
    year,
    month,
    dayofmonth,
)


BRONZE_PATH = (
    "s3a://kafka-spark-stock-project-kris/"
    "stock-market/bronze/rest/quote_snapshots/"
)

SILVER_PATH = (
    "s3a://kafka-spark-stock-project-kris/"
    "stock-market/silver/rest/quote_snapshots/"
)


def main():

    spark = (
        SparkSession.builder
        .appName("QuoteSnapshotsSilverWriter")
        .config("spark.sql.session.timeZone", "UTC")
        .config("spark.sql.shuffle.partitions", "4")
        .config(
            "spark.hadoop.fs.s3a.aws.credentials.provider",
            "org.apache.hadoop.fs.s3a.auth.IAMInstanceCredentialsProvider",
        )
        .getOrCreate()
    )

    spark.sparkContext.setLogLevel("WARN")

    bronze_df = (
        spark.read
        .option("recursiveFileLookup", "true")
        .json(BRONZE_PATH)
    )

    exploded_df = (
        bronze_df
        .withColumn(
            "quote",
            explode(col("records"))
        )
    )

    silver_df = (
        exploded_df

        .select(
            "batch_id",
            "schema_version",
            "source",

            to_timestamp(
                col("collected_at_utc")
            ).alias("collected_at"),

            col("quote.symbol")
                .alias("symbol"),

            col("quote.current_price")
                .cast("double")
                .alias("current_price"),

            col("quote.change")
                .cast("double")
                .alias("change"),

            col("quote.percent_change")
                .cast("double")
                .alias("percent_change"),

            col("quote.day_open")
                .cast("double")
                .alias("day_open"),

            col("quote.day_high")
                .cast("double")
                .alias("day_high"),

            col("quote.day_low")
                .cast("double")
                .alias("day_low"),

            col("quote.previous_close")
                .cast("double")
                .alias("previous_close"),

            col("quote.quote_timestamp")
                .cast("long")
                .alias("quote_timestamp_unix"),
        )

        .withColumn(
            "quote_timestamp",
            to_timestamp(
                from_unixtime(
                    col("quote_timestamp_unix")
                )
            )
        )

        .withColumn(
            "snapshot_date",
            to_date(col("collected_at"))
        )

        .withColumn(
            "year",
            year(col("collected_at"))
        )

        .withColumn(
            "month",
            month(col("collected_at"))
        )

        .withColumn(
            "day",
            dayofmonth(col("collected_at"))
        )
    )

    print(
        f"Silver quote records: {silver_df.count()}"
    )

    (
        silver_df.write
        .mode("overwrite")
        .option("compression", "snappy")
        .partitionBy(
            "year",
            "month",
            "day"
        )
        .parquet(SILVER_PATH)
    )

    print("=" * 60)
    print("Quote Silver write completed")
    print(f"Silver path: {SILVER_PATH}")
    print("=" * 60)

    spark.stop()


if __name__ == "__main__":
    main()